In [ ]:
!pip install transformers
!pip install torch


In [ ]:
!pip install datasets pandas matplotlib seaborn spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from datasets import load_dataset
import pandas as pd
import re
import spacy
import json

In [ ]:
path = "/content/FinEntity.json" # path to finEntity file
df = pd.read_json(path)
print(df)
print(df.columns)

                                               content  \
0    Johnson & Johnson <JNJ.N> shares gained 0.20% ...   
1    On the positive side, Siemens is rallying 6% a...   
2    Brent crude <LCOc1> rose 1.4% to $100.69 per b...   
3    Nearly all major S&P 500 sectors are red, with...   
4    NEW YORK - Wall Street ended sharply higher on...   
..                                                 ...   
979  Global stock markets traded in a narrow range ...   
980  Oil prices were little changed as traders awai...   
981  The central bank kept interest rates steady du...   
982  Bond yields remained flat as investors looked ...   
983  Gold prices edged higher while the dollar inde...   

                                           annotations  
0    [{'end': 17, 'tag': 'Positive', 'value': 'John...  
1    [{'end': 107, 'tag': 'Positive', 'value': 'Huh...  
2    [{'end': 11, 'tag': 'Positive', 'value': 'Bren...  
3    [{'end': 162, 'tag': 'Positive', 'value': 'hea...  
4    [{'end': 69, 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print(df.columns)

Index(['content', 'annotations'], dtype='object')


In [ ]:
data = []
for _, row in df.iterrows():
    sentence = row['content']

    if row['annotations']:  # if there are entities
        for ann in row['annotations']:
            entity = ann.get('value')
            sentiment = ann.get('tag')
            data.append([sentence, entity, sentiment])
    else:  # no entities
        data.append([sentence, "No financial entity found in the sequence", "none"])

df_flat = pd.DataFrame(data, columns=['sentence', 'entity', 'sentiment'])

print("\nFlattened dataset sample:\n", df_flat.tail(20))


Flattened dataset sample:
                                                sentence  \
2116  Several heavyweight private equity funds have ...   
2117  Several heavyweight private equity funds have ...   
2118  "We expect the wheat crop to be close to last ...   
2119  Chipmakers, including Infineon <IFXGn.DE> and ...   
2120  Chipmakers, including Infineon <IFXGn.DE> and ...   
2121  Coca-Cola Co <KO.N> rose 2.4% after the compan...   
2122  The Nikkei <.N225> rose 1.21% to 27,527.64, in...   
2123  The Nikkei <.N225> rose 1.21% to 27,527.64, in...   
2124  What is troubling some investors is that crack...   
2125  What is troubling some investors is that crack...   
2126  What is troubling some investors is that crack...   
2127  What is troubling some investors is that crack...   
2128  What is troubling some investors is that crack...   
2129  What is troubling some investors is that crack...   
2130  What is troubling some investors is that crack...   
2131  Global stock markets t

In [ ]:
test_cases = df_flat.sample(5, random_state=42)

print(test_cases)
examples = {
    "tokens": [sentence.split() for sentence in test_cases['sentence']],
    "labels": [["O"] * len(sentence.split()) for sentence in test_cases['sentence']]
}


                                               sentence               entity  \
1509  CME e-mini Nasdaq 100 futures <NQcv1> are off ...             Alphabet   
233   Refinitiv projected average U.S. gas demand in...            Refinitiv   
438   Stocks on Wall Street are down on Friday after...      Federal Reserve   
298   One of the first numbers investors look for in...                  UBS   
1896  Brent crude futures <LCOc1> rose 60 cents, or ...  Brent crude futures   

     sentiment  
1509  Negative  
233    Neutral  
438    Neutral  
298    Neutral  
1896  Positive  


In [ ]:
def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'<[A-Z]+\.[A-Z]+>', '', text)
    text = re.sub(r'[^A-Za-z0-9.,!?;\'\"\s]+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("\nBefore cleaning:\n", df_flat['sentence'].head(5))
df_flat['sentence'] = df_flat['sentence'].apply(clean_text)
print("\nAfter cleaning:\n", df_flat['sentence'].head(5))


Before cleaning:
 0    Johnson & Johnson <JNJ.N> shares gained 0.20% ...
1    On the positive side, Siemens is rallying 6% a...
2    On the positive side, Siemens is rallying 6% a...
3    Brent crude <LCOc1> rose 1.4% to $100.69 per b...
4    Brent crude <LCOc1> rose 1.4% to $100.69 per b...
Name: sentence, dtype: object

After cleaning:
 0    Johnson Johnson shares gained 0.20 after posti...
1    On the positive side, Siemens is rallying 6 af...
2    On the positive side, Siemens is rallying 6 af...
3    Brent crude rose 1.4 to 100.69 per barrel and ...
4    Brent crude rose 1.4 to 100.69 per barrel and ...
Name: sentence, dtype: object


In [ ]:
nlp = spacy.load("en_core_web_sm")

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

df_flat['sentence'] = df_flat['sentence'].str.lower().apply(lemmatize_text)

In [ ]:
before_drop = len(df_flat)
df_flat.dropna(subset=['sentence'], inplace=True)
df_flat = df_flat[df_flat['sentence'].str.strip() != ""]
after_drop = len(df_flat)
print(f"\nDropped {before_drop - after_drop} empty rows. Remaining rows: {after_drop}")


Dropped 0 empty rows. Remaining rows: 2136


In [ ]:
ENTITY_TYPES = [
    "COMPANY",
    "ORGANIZATION",
    "CURRENCY",
    "PERSON",
    "OTHER"
]

In [ ]:
def get_entity_type(entity):
    entity_lower = entity.lower()

    if "inc" in entity_lower or "ltd" in entity_lower or "corp" in entity_lower:
        return "COMPANY"
    elif any(keyword in entity_lower for keyword in ["organization", "association", "committee", "foundation", "agency"]):
        return "ORGANIZATION"
    elif entity_lower in ["usd", "eur", "gbp", "inr"]:
        return "CURRENCY"
    elif all(word.isalpha() and word.istitle() for word in entity.split()):
        return "PERSON"
    else:
        return "OTHER"

df_flat['entity_type'] = df_flat['entity'].apply(lambda x: get_entity_type(str(x)))

In [ ]:
def get_entity_type_info(entity):
    entity_type = get_entity_type(entity)
    return entity_type

entity = "World Health Organization"
etype = get_entity_type_info(entity)
print(f"Entity: {entity}")
print(f"Entity Type: {etype}")

Entity: World Health Organization
Entity Type: ORGANIZATION


In [ ]:
SENTIMENTS = ["positive", "negative", "neutral"]
BILU = ["B", "I", "L","U"]

all_labels = ["O"]
for ent_type in ENTITY_TYPES:
    for sent in SENTIMENTS:
        for bilu in BILU:
            all_labels.append(f"{bilu}-{ent_type}-{sent}")

label2id = {label: idx for idx, label in enumerate(all_labels)}
id2label = {idx: label for label, idx in label2id.items()}
entity_type2id = {etype: idx for idx, etype in enumerate(ENTITY_TYPES)}
id2entity_type = {idx: etype for etype, idx in entity_type2id.items()}
mappings = {
    "label2id": label2id,                     # entity-sentiment combined label
    "id2label": id2label,
    "entity_type2id": entity_type2id,         # only entity types
    "id2entity_type": id2entity_type
}

print("\nSample label2id mapping (first 10):", dict(list(label2id.items())[:10]))
print(f"Total Labels: {len(all_labels)}")


Sample label2id mapping (first 10): {'O': 0, 'B-COMPANY-positive': 1, 'I-COMPANY-positive': 2, 'L-COMPANY-positive': 3, 'U-COMPANY-positive': 4, 'B-COMPANY-negative': 5, 'I-COMPANY-negative': 6, 'L-COMPANY-negative': 7, 'U-COMPANY-negative': 8, 'B-COMPANY-neutral': 9}
Total Labels: 61


In [ ]:
output_path = "Preprocessed_FinEntity_Flattened.csv"
df_flat.to_csv(output_path, index=False)
print(f"\nPreprocessing complete! Saved to {output_path}")

with open("label_mappings.json", "w") as f:
    json.dump(mappings, f, indent=4)
print("\nLabel schema & mappings saved to label_mappings.json")

output_path = "Preprocessed_FinEntity.csv"
df.to_csv(output_path, index=False)
print(f"\nPreprocessing complete! Saved to {output_path}")


Preprocessing complete! Saved to Preprocessed_FinEntity_Flattened.csv

Label schema & mappings saved to label_mappings.json

Preprocessing complete! Saved to Preprocessed_FinEntity.csv


In [ ]:
from transformers import AutoTokenizer
finbert_model_name = "ProsusAI/finbert"
FinBERT_tokenizer = AutoTokenizer.from_pretrained(finbert_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
def get_label2id(entity_type_with_label):
    if(entity_type_with_label == "OTHER"):
                return label2id["OTHER"]
    else:
        return label2id["U"+"-" +entity_type_with_label]

In [ ]:
def tokenize_and_align_labels(row):

    # Extract content from row
    content = str(row['content'])
    print(f"Sentence:\n{content}\n")

    # Tokenize the content
    tokenized_output = FinBERT_tokenizer(content, return_offsets_mapping = True, padding = True, truncation=True)

    # Initialize labels with id of "O" expect begenning and ending of sentence
    # labels values will change based on entities
    labels = [label2id["O"]] * len(tokenized_output.input_ids)
    labels[0] = -100
    labels[-1] =-100

    # input IDs and attention Mask
    input_ids = tokenized_output.input_ids
    attention_mask = tokenized_output.attention_mask
    # Get tokens and offsets from the tokenizer's output
    tokens = FinBERT_tokenizer.convert_ids_to_tokens(tokenized_output["input_ids"])
    offsets = tokenized_output["offset_mapping"]
    print(f"tokens:\n{tokens}\n")


    # Loop through each annotation in the row
    for annotation in row['annotations']:
        # handle empty annotation
        if not annotation:
            continue
        # Extract the entity value and get its type from the lookup function
        entity_value = annotation['value']
        entity_type = get_entity_type(entity_value)
        sentiment = annotation['tag'].lower()


        # Get the character start and end from the annotation
        offset_mappings_from_annotation = [annotation['start'] , annotation['end']]

        # Create a list to store tokens that belong to the current entity
        labels_to_change = []
        # Loop through the tokenizer's offsets to find matching tokens
        for offset in range(len(offsets)):
            # Check if the token's character range is within the annotation's range
            if(offsets[offset][0] >= offset_mappings_from_annotation[0] and offsets[offset][1] <= offset_mappings_from_annotation[1]):
                # If there's a match, get the token text from the content
                token_text = content[offsets[offset][0]:offsets[offset][1]]
                # Add the token text to the list if it's not an empty string
                if(token_text not in [""]):
                    labels_to_change.append(offset)

        # nothing to change
        if not labels_to_change:
            continue

        if(len(labels_to_change)==1):
            # single token entity so U tag
            labels[labels_to_change[0]] = label2id[f"U-{entity_type}-{sentiment}"]
        else:
            # multi - token enities so use B,I,L tags
            labels[labels_to_change[0]] = label2id[f"B-{entity_type}-{sentiment}"]

            for i in range(1, len(labels_to_change)-1):
                labels[labels_to_change[i]] = label2id[f"I-{entity_type}-{sentiment}"]

            labels[labels_to_change[len(labels_to_change)-1]] = label2id[f"L-{entity_type}-{sentiment}"]

    return {"input_ids":input_ids, "attention_mask":attention_mask, "labels": labels}



In [ ]:
print("SAMPLE OUTPUTS")
test_data = df[:5]
for i, row in test_data.iterrows():
    # print("*"*10)
    output = tokenize_and_align_labels(row)
    for k in output:
        print(f"{k}:\n{output[k]}")
        print()
    print("*"*10)

SAMPLE OUTPUTS
Sentence:
Johnson & Johnson <JNJ.N> shares gained 0.20% after posting results that beat expectations but cut its full-year outlook, citing a stronger dollar. [nL4N2Z028U]

tokens:
['[CLS]', 'johnson', '&', 'johnson', '<', 'j', '##n', '##j', '.', 'n', '>', 'shares', 'gained', '0', '.', '20', '%', 'after', 'posting', 'results', 'that', 'beat', 'expectations', 'but', 'cut', 'its', 'full', '-', 'year', 'outlook', ',', 'citing', 'a', 'stronger', 'dollar', '.', '[', 'nl', '##4', '##n', '##2', '##z', '##0', '##28', '##u', ']', '[SEP]']

input_ids:
[101, 3779, 1004, 3779, 1026, 1046, 2078, 3501, 1012, 1050, 1028, 6661, 4227, 1014, 1012, 2322, 1003, 2044, 14739, 3463, 2008, 3786, 10908, 2021, 3013, 2049, 2440, 1011, 2095, 17680, 1010, 8951, 1037, 6428, 7922, 1012, 1031, 17953, 2549, 2078, 2475, 2480, 2692, 22407, 2226, 1033, 102]

attention_mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
import random

print(df.columns)

def test_random_samples(n=5, seed=42):
    """Pick n random rows from df and test alignment on each."""
    random.seed(seed)
    idxs = random.sample(range(len(df)), k=n)

    for idx in idxs:
        row = df.iloc[idx]
        print(f"\n===== Row {idx} =====")
        # test_alignment(row)   # uses the pretty-printer we defined earlier
        output = tokenize_and_align_labels(row)
        print(output)
        print("="*40)


Index(['content', 'annotations'], dtype='object')


In [ ]:
test_random_samples(n=5)


===== Row 654 =====
Sentence:
Also taking centre stage will be the Bank of England. The market is fully priced for a rate hike of 75 basis points to its highest since late 2008 at 3.0%. <0#BOEWATCH> 

tokens:
['[CLS]', 'also', 'taking', 'centre', 'stage', 'will', 'be', 'the', 'bank', 'of', 'england', '.', 'the', 'market', 'is', 'fully', 'priced', 'for', 'a', 'rate', 'hike', 'of', '75', 'basis', 'points', 'to', 'its', 'highest', 'since', 'late', '2008', 'at', '3', '.', '0', '%', '.', '<', '0', '#', 'bo', '##ew', '##at', '##ch', '>', '[SEP]']

{'input_ids': [101, 2036, 2635, 2803, 2754, 2097, 2022, 1996, 2924, 1997, 2563, 1012, 1996, 3006, 2003, 3929, 21125, 2005, 1037, 3446, 21857, 1997, 4293, 3978, 2685, 2000, 2049, 3284, 2144, 2397, 2263, 2012, 1017, 1012, 1014, 1003, 1012, 1026, 1014, 1001, 8945, 7974, 4017, 2818, 1028, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], '

In [ ]:
id2label = {v: k for k, v in label2id.items()}

def test_alignment(row):
    aligned = tokenize_and_align_labels(row)
    tokens = FinBERT_tokenizer.convert_ids_to_tokens(aligned["input_ids"])
    labels = aligned["labels"]

    print("Sentence:", row['content'], "\n")
    for t, l in zip(tokens, labels):
        label_name = "IGN" if l == -100 else id2label.get(l, str(l))
        print(f"{t:15} {label_name}")


In [ ]:
# Multi-token entity
row = df[df['content'].str.contains("Johnson & Johnson")].iloc[0]
test_alignment(row)

# Multiple entities
# row = df[[len(a) > 1 for a in df['annotations']]].iloc[[0]]
# test_alignment(row)
no_ent_idxs = [i for i, anns in enumerate(df['annotations']) if len(anns) == 0]
print("No-entity rows:", no_ent_idxs)

no_ent_idxs = [i for i, anns in enumerate(df['annotations']) if len(anns) == 0]

if no_ent_idxs:
    row = df.iloc[no_ent_idxs[0]]
    print("Testing no-entity row")
    test_alignment(row)
else:
    print("No rows without entities found in dataset. Skipping test.")




Sentence:
Johnson & Johnson <JNJ.N> shares gained 0.20% after posting results that beat expectations but cut its full-year outlook, citing a stronger dollar. [nL4N2Z028U]

tokens:
['[CLS]', 'johnson', '&', 'johnson', '<', 'j', '##n', '##j', '.', 'n', '>', 'shares', 'gained', '0', '.', '20', '%', 'after', 'posting', 'results', 'that', 'beat', 'expectations', 'but', 'cut', 'its', 'full', '-', 'year', 'outlook', ',', 'citing', 'a', 'stronger', 'dollar', '.', '[', 'nl', '##4', '##n', '##2', '##z', '##0', '##28', '##u', ']', '[SEP]']

Sentence: Johnson & Johnson <JNJ.N> shares gained 0.20% after posting results that beat expectations but cut its full-year outlook, citing a stronger dollar. [nL4N2Z028U] 

[CLS]           IGN
johnson         B-OTHER-positive
&               I-OTHER-positive
johnson         L-OTHER-positive
<               O
j               O
##n             O
##j             O
.               O
n               O
>               O
shares          O
gained          O
0         

In [ ]:
# Create synthetic no-entity rows
synthetic_no_entities = pd.DataFrame([
    {"content": "Global stock markets traded in a narrow range on Monday amid investor caution.", "annotations": []},
    {"content": "Oil prices were little changed as traders awaited the outcome of the OPEC meeting.", "annotations": []},
    {"content": "The central bank kept interest rates steady during its latest policy review.", "annotations": []},
    {"content": "Bond yields remained flat as investors looked for more economic data.", "annotations": []},
    {"content": "Gold prices edged higher while the dollar index was largely unchanged.", "annotations": []}
])

# Test alignment on these
for i in range(len(synthetic_no_entities)):
    row = synthetic_no_entities.iloc[i]
    print(f"\n=== Synthetic No-Entity Row {i} ===")
    test_alignment(row)



=== Synthetic No-Entity Row 0 ===
Sentence:
Global stock markets traded in a narrow range on Monday amid investor caution.

tokens:
['[CLS]', 'global', 'stock', 'markets', 'traded', 'in', 'a', 'narrow', 'range', 'on', 'monday', 'amid', 'investor', 'caution', '.', '[SEP]']

Sentence: Global stock markets traded in a narrow range on Monday amid investor caution. 

[CLS]           IGN
global          O
stock           O
markets         O
traded          O
in              O
a               O
narrow          O
range           O
on              O
monday          O
amid            O
investor        O
caution         O
.               O
[SEP]           IGN

=== Synthetic No-Entity Row 1 ===
Sentence:
Oil prices were little changed as traders awaited the outcome of the OPEC meeting.

tokens:
['[CLS]', 'oil', 'prices', 'were', 'little', 'changed', 'as', 'traders', 'awaited', 'the', 'outcome', 'of', 'the', 'op', '##ec', 'meeting', '.', '[SEP]']

Sentence: Oil prices were little changed as trader

In [ ]:
# df = pd.read_csv("/content/Preprocessed_FinEntity.csv") # path to preprocessed_Finentity.csv
# with open("label_mappings.json", "r") as f:
#     mappings = json.load(f)

# label2id = mappings["label2id"]
# id2label = {int(k): v for k, v in mappings["id2label"].items()}
# print(df)

In [ ]:
print(df.columns)

Index(['content', 'annotations'], dtype='object')


In [ ]:
from datasets import Dataset, DatasetDict

hf_dataset = Dataset.from_pandas(df)

# Now you can apply your tokenization function using `map`
hf_dataset = hf_dataset.map(tokenize_and_align_labels)

dataset_split = hf_dataset.train_test_split(test_size=0.2, seed=42)
val_test_split = dataset_split["test"].train_test_split(test_size=0.5, seed=42)

final_datasets = DatasetDict({
    "train": dataset_split["train"],
    "validation": val_test_split["train"],
    "test": val_test_split["test"]
})

Map:   0%|          | 0/984 [00:00<?, ? examples/s]

Streaming output truncated to the last 5000 lines.
['[CLS]', 'but', 'then', 'it', 'became', 'apparent', 'this', 'week', 'that', 'all', 'of', 'retail', 'was', 'not', 'uniformly', 'miserable', ',', 'with', 'companies', 'including', 'nord', '##strom', '<', 'j', '##wn', '.', 'n', '>', 'and', 'macy', "'", 's', '<', 'm', '.', 'n', '>', 'releasing', 'decent', 'numbers', 'and', 'upbeat', 'financial', 'forecast', '##s', '.', 'williams', '-', 'son', '##oma', '<', 'w', '##sm', '.', 'n', '>', ',', 'up', '13', '.', '9', '%', ',', 'also', 'beat', 'quarterly', 'revenue', 'expectations', 'on', 'strong', 'demand', 'for', 'its', 'furniture', 'and', 'home', 'improvement', 'goods', '.', '[', 'nl', '##3', '##n', '##2', '##x', '##h', '##3', '##f', '##0', ']', '[SEP]']

Sentence:
Worsening the mood, Russian President Vladimir Putin put nuclear-armed forces on high alert on Sunday. [nL1N2V308I]    Defense stocks L3Harris Technologies <LHX.N>, Raytheon Technologies <RTX.N>, Lockheed Martin Corp <LMT.N>, Genera

In [ ]:
import os
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# Load the model

os.environ["WANDB_DISABLED"] = "true"
model = AutoModelForTokenClassification.from_pretrained(
    finbert_model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
print("\n✅ Model loaded successfully with a new token classification head.")

# Define the function to compute metrics
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Flatten lists and ignore -100
    true_predictions = [
        id2label[p]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]
    true_labels = [
        id2label[l]
        for prediction, label in zip(predictions, labels)
        for p, l in zip(prediction, label)
        if l != -100
    ]

    return {
        "accuracy": accuracy_score(true_labels, true_predictions),
        "precision": precision_score(true_labels, true_predictions, average="weighted"),
        "recall": recall_score(true_labels, true_predictions, average="weighted"),
        "f1": f1_score(true_labels, true_predictions, average="weighted")
    }

# Define Training Arguments
training_args = TrainingArguments(
    output_dir="./finbert_ner_results",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to = "none",
)

# Initialize the Trainer
data_collator = DataCollatorForTokenClassification(tokenizer=FinBERT_tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    tokenizer=FinBERT_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Run Training
print("\n🚀 Starting model training...")
trainer.train()
print("🏁 Training complete.")

# Initial Evaluation
print("\n📊 Evaluating model on the validation set...")
evaluation_results = trainer.evaluate()

print("\n--- Initial Evaluation Report ---")
for key, value in evaluation_results.items():
    print(f"{key}: {value:.4f}")
print("---------------------------------")

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ProsusAI/finbert and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([61, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([61]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1907589093.py:69: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(



✅ Model loaded successfully with a new token classification head.

🚀 Starting model training...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator

🏁 Training complete.

📊 Evaluating model on the validation set...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)



--- Initial Evaluation Report ---
eval_loss: 0.1167
eval_accuracy: 0.9626
eval_precision: 0.9643
eval_recall: 0.9626
eval_f1: 0.9624
eval_runtime: 19.8020
eval_samples_per_second: 4.9490
eval_steps_per_second: 0.6570
epoch: 10.0000
---------------------------------


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
trainer.save_model("./finbert_ner_model")
FinBERT_tokenizer.save_pretrained("./finbert_ner_model")

('./finbert_ner_model/tokenizer_config.json',
 './finbert_ner_model/special_tokens_map.json',
 './finbert_ner_model/vocab.txt',
 './finbert_ner_model/added_tokens.json',
 './finbert_ner_model/tokenizer.json')

In [ ]:
!cp -r /content/finbert_ner_model /content/drive/MyDrive/
!cp -r /content/finbert_ner_results /content/drive/MyDrive/

^C


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

model_path = "./finbert_ner_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)
